# Negocio: de los datos a la recomendación municipal

Este notebook explica, paso a paso, cómo se transforma la Gold municipal en una recomendación de aperturas para un gimnasio low-cost.

Se comparan tres estrategias:

1. **Baseline**: los cinco municipios con mayor población de 18 a 50 años.
2. **Modelo**: municipios con menor competencia relativa que la media de su cluster, viables y que superan el control de calidad.
3. **Modelo diversificado por clusters**: alternativa que intenta representar distintos perfiles municipales, sin introducir municipios no viables o con calidad insuficiente.

El notebook **no redefine las fórmulas**. Utiliza las funciones de `src/negocio/valor_negocio.py`. Las cifras son estimaciones bajo supuestos centrales, no ingresos observados.

La ejecución automática que genera CSV y `resumen_negocio.md` sigue siendo:

```powershell
python -m src.negocio.ejecutar
```

## 0. Configuración

Se localiza la raíz del repositorio y se importan los módulos oficiales. El notebook se puede abrir desde la raíz o desde `notebooks/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
project_root = next(
    (
        path for path in (working_directory, *working_directory.parents)
        if (path / "src" / "negocio" / "ejecutar.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("No se encontró la raíz con src/negocio/ejecutar.py")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.negocio import ejecutar, valor_negocio
from src.negocio.supuestos import ESCENARIOS_PESOS, PESOS_CENTRAL, SUPUESTOS, parametros
from src.procesado.common.paths import GOLD_TABLE_PATH

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print(f"Raíz del proyecto: {project_root}")

## 1. Carga y validación de entradas

Se leen la Gold y los clusters mediante la función oficial `ejecutar.cargar()`.

Comprobaciones principales:

- 785 municipios.
- Un código municipal único de cinco dígitos.
- Un cluster informado para cada municipio.
- Recuentos de competencia disponibles.

In [ ]:
datos = ejecutar.cargar()

assert len(datos) == 785, f"Se esperaban 785 municipios y hay {len(datos)}"
assert datos["municipality_code"].is_unique
assert datos["municipality_code"].astype("string").str.fullmatch(r"\d{5}").all()
assert datos["cluster"].notna().all()

print(f"Gold + clusters: {len(datos)} municipios")
print(f"Origen Gold: {GOLD_TABLE_PATH}")
display(
    datos[[
        "municipality_code", "municipality_name", "province",
        "population_18_50", "gyms_count",
        "other_sports_facilities_count", "cluster",
    ]].head(8)
)
display(
    datos.groupby("cluster").size().rename("municipios").to_frame()
)

## 2. Supuestos del escenario central

El cálculo central utiliza los valores definidos en `supuestos.py`. Las fuentes completas se mantienen en la documentación del proyecto.

Los costes considerados suman **556.000 € anuales**. Como referencia operativa se plantea un centro estándar con capacidad máxima de **2.600 socios**. Con un ticket mensual sin IVA de **22,37 €**, un único centro tendría una facturación operativa máxima de:

\[
2.600 	imes 22,37 	imes 12 = 697.944\;€ / año
\]

Esta capacidad no limita el potencial del municipio. Cuando la demanda captable supera ampliamente 2.600 socios, puede estudiarse un centro mayor, varias aperturas o una configuración económica diferente.

In [ ]:
p = parametros("central")
claves = [
    "penetracion", "ticket_mensual", "capacidad", "cuota_max",
    "atractivo_nuevo", "coste_anual_club",
]
filas = []
for clave in claves:
    info = SUPUESTOS.get(clave, {})
    filas.append({
        "Supuesto": clave,
        "Valor central": p[clave],
        "Fuente / criterio": info.get("fuente", "Ver documentación"),
    })
display(pd.DataFrame(filas))

display(pd.DataFrame([
    {"Tipo de instalación": tipo, "Peso": peso}
    for tipo, peso in ESCENARIOS_PESOS[PESOS_CENTRAL].items()
]))

facturacion_operativa_maxima = p["capacidad"] * p["ticket_mensual"] * 12
print(f"Facturación operativa máxima del centro estándar: {facturacion_operativa_maxima:,.0f} € / año")

## 3. Competencia ponderada

La competencia se transforma a **equivalentes de gimnasio**:

\[
competidores\_pond = gimnasios 	imes 1 + otras\ instalaciones 	imes 0,1
\]

Después se normaliza por población objetivo:

\[
competencia\ por\ 10.000 = rac{competidores\_pond}{población\ 18{-}50} 	imes 10.000
\]

La ponderación permite que una instalación deportiva distinta de un gimnasio influya en la competencia, pero con menor peso.

In [ ]:
competencia = valor_negocio.competidores_ponderados(datos, pesos=PESOS_CENTRAL)

muestra_nombres = [
    "Sevilla", "Málaga", "Marbella", "Algeciras",
    "Huelva", "Fuengirola", "Puerto de Santa María, El",
]
muestra_competencia = competencia.loc[
    competencia["municipality_name"].isin(muestra_nombres),
    [
        "municipality_name", "population_18_50", "gyms_count",
        "other_sports_facilities_count", "competidores_pond",
        "comp_per_10k_18_50",
    ],
].sort_values("population_18_50", ascending=False)
display(muestra_competencia)

## 4. Control de calidad para todos los municipios

El control de calidad no busca huecos de mercado. Comprueba si la competencia registrada parece suficientemente completa.

Se utiliza como referencia **1 equivalente de gimnasio por cada 10.000 habitantes de 18 a 50 años**:

\[
expected\_gyms = rac{población\ 18{-}50}{10.000}
\]

\[
weighted\_observed\_vs\_expected\_pct =
rac{competidores\_pond}{expected\_gyms} 	imes 100
\]

- `quality_flag=True`: la competencia ponderada alcanza al menos el 50 % de la referencia.
- `quality_flag=False`: el posible hueco debe validarse con una segunda fuente.
- Un porcentaje superior al 100 % es posible: significa que la oferta ponderada observada supera la referencia. **No demuestra por sí solo sobresaturación ni un error.**

Ejemplos: un 81 % supera el control; un 215 % equivale a 2,15 veces la referencia.

In [ ]:
columnas_calidad = [
    "municipality_name", "competidores_pond", "expected_gyms",
    "weighted_observed_vs_expected_pct", "quality_flag", "quality_status",
]
display(
    competencia.loc[
        competencia["municipality_name"].isin(muestra_nombres),
        columnas_calidad,
    ].sort_values("weighted_observed_vs_expected_pct", ascending=False)
)

print("Distribución del control de calidad:")
display(
    competencia["quality_flag"].value_counts(dropna=False)
    .rename_axis("quality_flag").rename("municipios").to_frame()
)

## 5. Candidatos: hueco relativo dentro del cluster

El criterio de candidatura responde a una pregunta diferente:

> ¿Tiene el municipio menos competencia que otros municipios con un perfil parecido?

\[
candidate\_flag =
competencia\ por\ 10.000 < media\ de\ competencia\ del\ cluster
\]

Por tanto:

- `quality_flag` evalúa la confianza mínima en el dato.
- `candidate_flag` identifica un posible hueco relativo de mercado.

Un municipio solo puede entrar en la recomendación final si es candidato, viable y supera calidad.

In [ ]:
candidatos = valor_negocio.candidatos(competencia)

resumen_candidatos = candidatos.groupby("cluster").agg(
    municipios=("municipality_code", "size"),
    candidatos=("candidate_flag", "sum"),
    media_competencia=("media_cluster_comp_per_10k", "first"),
).reset_index()
display(resumen_candidatos)

display(
    candidatos[[
        "municipality_name", "cluster", "comp_per_10k_18_50",
        "media_cluster_comp_per_10k", "candidate_flag", "quality_flag",
    ]].head(10)
)

## 6. Demanda potencial

\[
socios\_potenciales = población\ 18{-}50 	imes penetración
\]

\[
demanda\ por\ competidor =
rac{socios\_potenciales}{\max(competidores\_pond, 1)}
\]

Es una estimación de mercado direccionable. No representa altas observadas.

In [ ]:
demanda = valor_negocio.demanda(candidatos, p)
display(
    demanda[[
        "municipality_name", "population_18_50", "socios_potenciales",
        "mercado_potencial_eur", "demanda_por_comp",
    ]].sort_values("demanda_por_comp", ascending=False).head(10)
)

## 7. Captación y facturación

La cuota del nuevo centro se estima mediante su atractivo relativo:

\[
cuota\_centro = \min\left(rac{f}{competidores\_pond + f}, cuota\_max
ight)
\]

\[
socios\_captables = socios\_potenciales 	imes cuota\_centro
\]

La **facturación anual sin tope** muestra el valor de la demanda captable y sirve para comparar municipios. La facturación con capacidad se conserva como referencia operativa de un centro estándar.

Cuando todos los seleccionados alcanzan 2.600 socios, no se repite en la tabla una facturación con tope idéntica ni el mismo margen. Ese resultado se resume aparte.

In [ ]:
facturacion = valor_negocio.facturacion(demanda, p, regla="C")

display(
    facturacion[[
        "municipality_name", "cuota_centro", "socios_captables",
        "facturacion_sin_tope_capacidad", "ratio_capacidad",
        "exceso_sobre_capacidad",
    ]].sort_values("facturacion_sin_tope_capacidad", ascending=False).head(10)
)

## 8. Viabilidad económica

\[
margen = facturación\ operativa - coste\ anual
\]

Un municipio es viable cuando el margen operativo estimado es igual o superior a cero. El cálculo no incluye CAPEX, financiación, amortización ni impuestos.

In [ ]:
evaluacion = valor_negocio.viabilidad(facturacion, p)

display(
    evaluacion[[
        "municipality_name", "socios_break_even", "facturacion_anual",
        "margen_eur", "colchon_demanda_pct", "viable",
    ]].sort_values("margen_eur", ascending=False).head(10)
)
print(f"Municipios viables: {int(evaluacion['viable'].sum())} de {len(evaluacion)}")

## 9. Baseline: práctica simple del cliente

El baseline selecciona los cinco municipios con mayor población de 18 a 50 años, sin filtrar por competencia, viabilidad o calidad. Después se evalúa con las mismas fórmulas económicas que el modelo.

In [ ]:
baseline = valor_negocio.top5_baseline(evaluacion)
columnas_baseline = [
    "municipality_name", "cluster", "population_18_50", "competidores_pond",
    "facturacion_anual", "facturacion_sin_tope_capacidad",
    "margen_eur", "viable", "quality_flag",
    "weighted_observed_vs_expected_pct", "quality_status",
]
display(baseline[columnas_baseline].reset_index(drop=True))

## 10. Modelo: selección final y sustituciones por calidad

El ranking económico se construye entre candidatos viables. Si alguno no supera calidad, no desaparece: pasa a la tabla de oportunidades pendientes y se sustituye por el siguiente municipio válido.

Esto garantiza que la recomendación final contiene hasta cinco municipios:

```text
candidate_flag=True
+ viable=True
+ quality_flag=True
```

In [ ]:
(
    modelo,
    pendientes_calidad,
    ranking_modelo_completo,
    sustituciones_calidad,
) = valor_negocio.seleccionar_modelo_con_calidad(evaluacion, n=5)

columnas_modelo = [
    "posicion", "municipality_name", "cluster", "population_18_50",
    "competidores_pond", "comp_per_10k_18_50",
    "media_cluster_comp_per_10k", "socios_captables",
    "facturacion_sin_tope_capacidad", "ratio_capacidad",
    "weighted_observed_vs_expected_pct", "quality_status",
]
display(Markdown("### Selección final del Modelo"))
display(modelo[[c for c in columnas_modelo if c in modelo.columns]].reset_index(drop=True))

if not modelo.empty and modelo["facturacion_anual"].nunique(dropna=False) == 1:
    fact_tope = modelo["facturacion_anual"].iloc[0]
    margen_tope = modelo["margen_eur"].iloc[0]
    print(
        f"Los {len(modelo)} municipios alcanzan la capacidad de referencia: "
        f"facturación operativa {fact_tope:,.0f} € y margen {margen_tope:,.0f} € por centro."
    )

### Oportunidades sustituidas o pendientes de validación

Estas oportunidades mantienen su posición económica original, pero no ocupan una plaza final hasta validar la competencia con una segunda fuente.

In [ ]:
columnas_pendientes = [
    "posicion_ranking_precalidad", "municipality_name", "cluster",
    "competidores_pond", "expected_gyms",
    "weighted_observed_vs_expected_pct",
    "facturacion_sin_tope_capacidad", "motivo_calidad", "quality_status",
]
display(pendientes_calidad[[c for c in columnas_pendientes if c in pendientes_calidad.columns]])

display(Markdown("### Sustituciones realizadas"))
if sustituciones_calidad.empty:
    print("No fue necesario sustituir municipios por calidad.")
else:
    display(sustituciones_calidad)

## 11. Modelo diversificado por clusters

Esta alternativa intenta representar distintos perfiles municipales, pero mantiene las mismas condiciones mínimas: candidatura, viabilidad y calidad.

Si queda vacía o representa menos clusters de los esperados, no es un error automático. Significa que algunos clusters no contienen municipios que cumplan simultáneamente las tres condiciones.

In [ ]:
diversificada = valor_negocio.top5_diversificado_clusters(evaluacion)

columnas_div = [
    "posicion", "municipality_name", "cluster", "population_18_50",
    "facturacion_sin_tope_capacidad", "ratio_capacidad",
    "weighted_observed_vs_expected_pct", "quality_status", "motivo_seleccion",
]
display(diversificada[[c for c in columnas_div if c in diversificada.columns]])

# Diagnóstico por cluster: dónde se pierden municipios.
diagnostico_clusters = evaluacion.groupby("cluster").agg(
    municipios=("municipality_code", "size"),
    candidatos=("candidate_flag", "sum"),
    viables=("viable", "sum"),
    calidad_ok=("quality_flag", "sum"),
).reset_index()

diagnostico_clusters["candidato_viable_calidad"] = evaluacion.assign(
    valido=(
        evaluacion["candidate_flag"]
        & evaluacion["viable"]
        & evaluacion["quality_flag"]
    )
).groupby("cluster")["valido"].sum().to_numpy()

display(Markdown("### Por qué algunos clusters no aportan municipios"))
display(diagnostico_clusters)

clusters_sin_validos = diagnostico_clusters.loc[
    diagnostico_clusters["candidato_viable_calidad"].eq(0), "cluster"
].tolist()
if clusters_sin_validos:
    print(
        "Clusters sin candidatos simultáneamente viables y con calidad suficiente: "
        + ", ".join(map(str, clusters_sin_validos))
    )
else:
    print("Todos los clusters tienen al menos un candidato viable con calidad suficiente.")

## 12. Comparación final: por qué el Modelo mejora el baseline

La comparación utiliza las mismas fórmulas económicas para ambas estrategias. La mejora no se demuestra solo por población o facturación, sino por combinar:

- hueco relativo dentro del cluster;
- viabilidad operativa;
- control de calidad de competencia;
- demanda captable antes del límite operativo.

In [ ]:
selecciones = {
    "baseline": baseline,
    "modelo_a": modelo,
    "diversificada": diversificada,
}

comparacion_baseline_modelo, cambios = valor_negocio.comparar(baseline, modelo)
display(Markdown("### Baseline frente a Modelo"))
display(comparacion_baseline_modelo)
print("Cambios de municipios:", cambios)

resumen_comparativo = pd.DataFrame([
    {
        "Estrategia": "Baseline",
        "Centros": len(baseline),
        "Viables": int(baseline["viable"].sum()),
        "Facturación operativa total": baseline["facturacion_anual"].sum(),
        "Margen total": baseline["margen_eur"].sum(),
        "Facturación sin tope total": baseline["facturacion_sin_tope_capacidad"].sum(),
        "Clusters representados": baseline["cluster"].nunique(),
    },
    {
        "Estrategia": "Modelo",
        "Centros": len(modelo),
        "Viables": int(modelo["viable"].sum()),
        "Facturación operativa total": modelo["facturacion_anual"].sum(),
        "Margen total": modelo["margen_eur"].sum(),
        "Facturación sin tope total": modelo["facturacion_sin_tope_capacidad"].sum(),
        "Clusters representados": modelo["cluster"].nunique(),
    },
    {
        "Estrategia": "Modelo diversificado",
        "Centros": len(diversificada),
        "Viables": int(diversificada["viable"].sum()) if not diversificada.empty else 0,
        "Facturación operativa total": diversificada["facturacion_anual"].sum(),
        "Margen total": diversificada["margen_eur"].sum(),
        "Facturación sin tope total": diversificada["facturacion_sin_tope_capacidad"].sum(),
        "Clusters representados": diversificada["cluster"].nunique() if not diversificada.empty else 0,
    },
])
display(resumen_comparativo)

## 13. Visualización de demanda y capacidad

El gráfico diferencia el potencial captable de la facturación operativa de un centro estándar. Una brecha grande indica demanda excedente, no ingresos garantizados.

In [ ]:
grafico_modelo = modelo.copy().sort_values("facturacion_sin_tope_capacidad")
if not grafico_modelo.empty:
    y = np.arange(len(grafico_modelo))
    plt.figure(figsize=(10, 5))
    plt.barh(y - 0.18, grafico_modelo["facturacion_sin_tope_capacidad"], height=0.36, label="Demanda captable sin tope")
    plt.barh(y + 0.18, grafico_modelo["facturacion_anual"], height=0.36, label="Referencia operativa con capacidad")
    plt.yticks(y, grafico_modelo["municipality_name"])
    plt.xlabel("EUR anuales estimados")
    plt.title("Potencial captable frente a referencia operativa")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 14. Sensibilidad y robustez

La robustez mide en cuántas combinaciones de supuestos aparece cada municipio entre los seleccionados:

\[
3\ penetraciones 	imes 3\ cuotas 	imes 3\ atractivos
	imes 2\ pesos 	imes 2\ reglas = 108\ combinaciones
\]

- Las apariciones **no son probabilidades**.
- La posición media se calcula solo cuando el municipio aparece.
- La facturación mínima es el resultado más conservador entre esos escenarios; no incluye aquellos en los que no fue seleccionado.
- La calidad mostrada pertenece al escenario central.
- El escenario extremo `todo_igual` está excluido.

In [ ]:
sensibilidad_detalle, sensibilidad_frecuencias = valor_negocio.sensibilidad(datos)

# Añadimos al resumen el estado de calidad fijo del escenario central.
calidad_central = evaluacion[["municipality_code", "quality_status"]].rename(
    columns={"quality_status": "quality_status_central"}
)
robustez = sensibilidad_frecuencias.merge(
    calidad_central, on="municipality_code", how="left", validate="one_to_one"
)

columnas_robustez = [
    "municipality_name", "apariciones", "posicion_media", "fact_min",
    "robustez", "quality_status_central",
]
display(robustez[columnas_robustez].head(10))
print(f"Combinaciones evaluadas: {robustez['combinaciones_totales'].max()}")

In [ ]:
grafico_robustez = robustez.head(10).sort_values("apariciones")
plt.figure(figsize=(9, 5))
plt.barh(grafico_robustez["municipality_name"], grafico_robustez["apariciones"])
plt.xlabel("Apariciones entre 108 combinaciones")
plt.title("Robustez de los municipios más frecuentes")
plt.tight_layout()
plt.show()

## 15. Validación final

El recorrido paso a paso debe coincidir con la evaluación compuesta oficial. También se comprueba que:

- el Modelo tiene como máximo cinco municipios;
- no existen duplicados;
- todos son candidatos, viables y superan calidad;
- la sensibilidad contiene 108 combinaciones;
- no se utiliza `todo_igual`.

In [ ]:
evaluacion_oficial = valor_negocio.evaluar(
    datos,
    escenario="central",
    pesos=PESOS_CENTRAL,
    regla="C",
    con_candidatos=True,
)
pd.testing.assert_frame_equal(evaluacion, evaluacion_oficial)

assert len(modelo) <= 5
assert modelo["municipality_code"].is_unique
assert (
    modelo["candidate_flag"]
    & modelo["viable"]
    & modelo["quality_flag"]
).all()
assert sensibilidad_frecuencias["combinaciones_totales"].eq(108).all()
assert not sensibilidad_detalle["escenario"].str.contains("todo_igual", na=False).any()

print("Validaciones del notebook: OK")

## 16. Conclusión y lectura de negocio

La recomendación final debe interpretarse así:

- El **baseline** representa la práctica simple de seleccionar por tamaño de mercado.
- El **Modelo** añade competencia relativa, viabilidad y calidad mínima del dato.
- Las oportunidades con calidad insuficiente no se borran; quedan pendientes de comprobar con una segunda fuente.
- El **Modelo diversificado** solo puede representar clusters que contengan candidatos simultáneamente viables y con calidad suficiente.
- La facturación anual sin tope permite diferenciar la demanda captable. La cifra de 697.944 € es únicamente la referencia operativa para un centro estándar de 2.600 socios.
- Antes de invertir deben validarse competencia real, ubicación concreta, costes locales, CAPEX y ritmo de maduración del centro.

Para generar todos los CSV y el informe ejecutivo:

```powershell
python -m src.negocio.ejecutar
```

In [ ]:
recomendados = ", ".join(modelo["municipality_name"].tolist()) or "ninguno"
descartados = ", ".join(
    pendientes_calidad.loc[
        pendientes_calidad["motivo_calidad"].str.startswith("Sustituido", na=False),
        "municipality_name",
    ].tolist()
) or "ninguno"

print(f"Municipios recomendados por el Modelo: {recomendados}")
print(f"Municipios sustituidos por calidad: {descartados}")
print(
    f"El Modelo selecciona {int(modelo['viable'].sum())} municipios viables de {len(modelo)}, "
    f"frente a {int(baseline['viable'].sum())} viables de {len(baseline)} en el baseline."
)
print(
    f"Clusters representados: Modelo={modelo['cluster'].nunique()}, "
    f"diversificado={diversificada['cluster'].nunique() if not diversificada.empty else 0}."
)